# Chest X-ray Lung Segmentation

ديمو للمقابلة يحدد الرئة اليمين والشمال على أشعة الصدر. هنستخدم **PSPNet pretrained** من [TorchXRayVision](https://github.com/mlmed/torchxrayvision) ونضيف مقارنة للـmasks عند عتبات مختلفة ومقاييس تشرح درجة تغيّر الحدود.

في Google Colab اعمل **Open notebook → Upload**، وبعدها شغّل الخلايا بالترتيب. محليًا افتح الملف في VS Code أو Jupyter واختار بيئة Python فيها Torch وTorchVision متوافقين. أول تشغيل بينزّل الكود لو مش موجود والـcheckpoint الأصلي، حوالي 260 MiB. بيشتغل على CPU. العرض تعليمي باستخدام صورة المثال المنشورة مع الريبو.

- **من الريبو:** الأوزان، PSPNet، ودوال تجهيز الصورة.
- **الإضافة:** مقارنة الـmasks، حساب نسبة مساحة البكسلات في كل mask، وDice مقابل عتبة 0.50 لقياس ثبات النتيجة لما نغيّر العتبة.


In [ ]:
from importlib import invalidate_caches
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/mlmed/torchxrayvision.git"
REPO_COMMIT = "17726dd8f5e756eec3e5ae958281a289b05b1dae"
XRV_VERSION = "1.5.7"
SAMPLE_FILE = Path("tests/covid-19-pneumonia-58-prior.jpg")
PACKAGE_INIT = Path("torchxrayvision/__init__.py")

root = Path.cwd()
new_checkout = root / "torchxrayvision-upstream"
candidates = (root, new_checkout, root / "torchxrayvision")
repo_dir = next(
    (path for path in candidates if (path / SAMPLE_FILE).is_file() and (path / PACKAGE_INIT).is_file()),
    new_checkout,
)
if not repo_dir.exists():
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(repo_dir)], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "checkout", "--quiet", REPO_COMMIT], check=True)

sample_path = repo_dir / SAMPLE_FILE
package_init = (repo_dir / PACKAGE_INIT).resolve()
if not sample_path.is_file() or not package_init.is_file():
    raise FileNotFoundError(f"Incomplete TorchXRayVision checkout at {repo_dir}")

try:
    installed_xrv = version("torchxrayvision")
except PackageNotFoundError:
    installed_xrv = None
if installed_xrv != XRV_VERSION:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"torchxrayvision=={XRV_VERSION}"], check=True)

# An earlier Colab run can have cached the repo's empty outer package.
loaded_module = sys.modules.get("torchxrayvision")
loaded_file = getattr(loaded_module, "__file__", None)
if loaded_module is not None and (loaded_file is None or Path(loaded_file).resolve() != package_init):
    for name in tuple(sys.modules):
        if name == "torchxrayvision" or name.startswith("torchxrayvision."):
            del sys.modules[name]

# Resolve the inner package directly, even if a legacy clone shadows it.
invalidate_caches()
sys.path.insert(0, str(repo_dir))
try:
    import torchxrayvision as xrv
finally:
    sys.path.pop(0)

print(f"Demo image: {sample_path}")
print(f"TorchXRayVision {xrv.__version__}: {xrv.__file__}")


## 1. تجهيز الأشعة

الصورة في المثال 8-bit. أول خطوة تقرأها وتحولها لقناة واحدة، ثم تغيّر مدى قيم الـpixels من `[0, 255]` إلى `[-1024, 1024]` بالشكل اللي مكتبة TorchXRayVision بتتوقعه. الـcenter crop بياخد مربع من النص، وبعده الـresize إلى `512×512`. نضيف بعدين بُعد الـbatch فالشكل النهائي `[1, 1, 512, 512]`، يعني صورة واحدة وقناة واحدة.

كل الـoverlays هتظهر على نفس الصورة المجهزة اللي دخلت الموديل. لو الصورة مش مربعة، الـcrop ممكن يشيل أجزاء من الحواف. وعشان كده بنعرض الصورة المجهزة جنب النتيجة.


In [ ]:
from time import perf_counter

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
from PIL import Image
import torch

with Image.open(sample_path) as source:
    image_rgb = np.asarray(source.convert("RGB"))

image_scaled = xrv.utils.normalize(image_rgb, 255)
image_single_channel = image_scaled.mean(axis=-1)[None, :, :]
image_cropped = xrv.datasets.XRayCenterCrop()(image_single_channel)
image_ready = xrv.datasets.XRayResizer(512)(image_cropped)
image_tensor = torch.from_numpy(image_ready).unsqueeze(0)
print(f"Input tensor: {tuple(image_tensor.shape)}")


## 2. تشغيل الموديل وفهم الناتج

PSPNet بيدمج معلومات من مناطق بأحجام مختلفة، فيقدر يستخدم تفاصيل محلية جنب سياق الصورة الأوسع. هنا الموديل متدرب مسبقًا على تقسيم **14 جزء تشريحي**. `eval()` يخليه يشتغل في وضع التقييم، و`inference_mode()` يوقف حسابات الـgradients اللي التدريب كان هيحتاجها.

شكل الناتج `[1, 14, 512, 512]`. الـ14 هنا خرائط كاملة، واحدة لكل جزء تشريحي. كل pixel له قيمة خام اسمها logit. بنستخدم sigmoid عشان نحولها لقيمة بين 0 و1 لكل خريطة بشكل مستقل. القيمة دي **model score**، ومع `score >= threshold` نطلع mask من True/False. بنختار الرئتين بأسمائهم من `model.targets`.


In [ ]:
# None uses the library's normal cache. You can use a local folder if needed.
WEIGHTS_CACHE = None
model = xrv.baseline_models.chestx_det.PSPNet(cache_dir=WEIGHTS_CACHE).eval()
structures = {"Left Lung": "#40c4ff", "Right Lung": "#ffab40"}
lung_indices = [model.targets.index(name) for name in structures]

with torch.inference_mode():
    started = perf_counter()
    logits = model(image_tensor)
    elapsed_ms = (perf_counter() - started) * 1000
    lung_scores = logits[0, lung_indices].sigmoid().cpu().numpy()

print(f"Model output: {tuple(logits.shape)}")
print(f"CPU forward pass: {elapsed_ms:.0f} ms")


## 3. التعديل اللي تضيفه وتعرضه

جرّب تغيّر `THRESHOLDS`. اللونين ثابتين عشان تتابع كل رئة. نسبة المساحة هي حصة الـmask من **بكسلات الصورة المجهزة**. تقليل العتبة يدخل pixels أكتر، ورفعها يخلّي الموديل يحتفظ بالبكسلات اللي ليها score أعلى.

بنحسب كمان Dice بين كل mask والـmask اللي طلعت عند 0.50. دي مقارنة ثبات بين مخرجات نفس الموديل. قياس الدقة يحتاج masks مرجعية رسمها مختصون أو مصدر بيانات موثوق.


In [ ]:
THRESHOLDS = (0.30, 0.50, 0.70)
REFERENCE_THRESHOLD = 0.50


def mask_dice(first, second):
    foreground_pixels = np.count_nonzero(first) + np.count_nonzero(second)
    if foreground_pixels == 0:
        return np.nan
    overlap = np.count_nonzero(first & second)
    return 2 * overlap / foreground_pixels


reference_masks = lung_scores >= REFERENCE_THRESHOLD
fig, axes = plt.subplots(1, len(THRESHOLDS) + 1, figsize=(3.3 * (len(THRESHOLDS) + 1), 4.0))
axes[0].imshow(image_ready[0], cmap="gray")
axes[0].set_title("Model input")
axes[0].axis("off")

print(f"{'Threshold':>9}  {'Anatomy':<11}  {'Image area':>10}  {'Dice vs 0.50':>12}")
for ax, threshold in zip(axes[1:], THRESHOLDS):
    ax.imshow(image_ready[0], cmap="gray")
    ax.set_title(f"Threshold {threshold:.2f}")
    ax.axis("off")
    masks = lung_scores >= threshold
    for index, (name, color) in enumerate(structures.items()):
        mask = masks[index]
        if mask.any() and not mask.all():
            ax.contour(mask.astype(np.float32), levels=[0.5], colors=[color], linewidths=1.7)
        stability = mask_dice(mask, reference_masks[index])
        print(f"{threshold:9.2f}  {name:<11}  {mask.mean():10.1%}  {stability:12.3f}")

legend = [Line2D([0], [0], color=color, linewidth=2, label=name) for name, color in structures.items()]
fig.legend(handles=legend, loc="lower center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()


## 4. شرح المقابلة

**ليه segmentation؟** التصنيف يرجع label على مستوى الصورة، والـsegmentation يرجع حدود العضو على مستوى الـpixel. تقدر تعرض الصورة والحدود جنب بعض وتراجعها بصريًا.

**أقيس الأداء إزاي؟** أقارن الـmasks مع ground truth على حالات متقسمّة حسب المريض. لو هختار threshold على الـvalidation، أثبّته قبل التقييم على test set. أبلّغ Dice وIoU لكل رئة وأراجع حالات الفشل على بيانات من أجهزة ومستشفيات مختلفة.

**إيه حدود الديمو؟** تشغيل حالة واحدة يثبت إن الـpipeline ماشية. تقييم التعميم محتاج بيانات مستقلة وتغطية اختلافات زي AP وPA وجودة الصورة. عيّنة الديمو صورة JPEG. دعم DICOM هيحتاج قراءة الـpixel data والتعامل مع الـmetadata والـphotometric interpretation مع الحفاظ على خصوصية بيانات الحالات.

**تقدّم الشغل إزاي؟**

> I built a chest X-ray segmentation demo on top of TorchXRayVision's pretrained PSPNet. I added a threshold-sensitivity comparison to show how the lung contours change. My next step would be to evaluate the masks against annotated cases from an independent dataset.

[Upstream notebook and sample](https://github.com/mlmed/torchxrayvision/blob/main/scripts/segmentation.ipynb) · [PSPNet model documentation](https://mlmed.org/torchxrayvision/models.html#chestx-det-segmentation) · [TorchXRayVision citation](https://github.com/mlmed/torchxrayvision/blob/main/CITATION)
